# Челленджи недели: классы, наследование, магические методы, dataclass

**Цель:** проверить, что концепты ООП работают вместе и встраиваются в инфраструктуру Python (`len`, `==`, `for`, `dict`-ключи, `print`).

**Как работать:**
- Часть задач сопровождается ячейкой «Объясни своими словами» — там нужно не только написать код, но и сформулировать, почему он работает.
- Задачи решаются на 5-25 строк кода. На W3 разрешены и `def`, и `class` — выбираем по задаче.
- Сначала попробуй сам, без подглядываний. Если застрял на 10+ минут — открой solution-версию.


## Задание 1: Класс `Vector` с арифметикой и хэшем

Реализуй класс `Vector` для двумерного вектора:

- `__init__(self, x, y)` — два координаты
- `__repr__` — формат `Vector(x=3, y=4)` (валидный Python-код)
- `__eq__` — два вектора равны, если совпадают `x` и `y`; для чужих типов верни `NotImplemented`
- `__hash__` — через `hash((self.x, self.y))`
- `__add__` — складывает два вектора покомпонентно, возвращает новый `Vector`

Проверка: `Vector(1, 2) + Vector(3, 4) == Vector(4, 6)` должно быть `True`, и оба объекта должны корректно работать как ключи `dict` или элементы `set`.

In [2]:

# TODO: implement
# your code:
class Vector():
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Vector(x={self.x}, y={self.y})"

    def __eq__(self, other):
        if isinstance(other, Vector):
            return (other.x == self.x and other.y == self.y)
        return NotImplemented

    def __hash__(self):
        return hash((self.x, self.y))

    def __add__(self, other):
        return Vector(self.x+other.x, self.y+other.y)

Vector(1, 2) + Vector(3, 4) == Vector(4, 6)

True

**Объясни своими словами:** почему нужно реализовывать `__hash__` вместе с `__eq__`, и что произойдёт, если оставить только `__eq__`?

чтобы сравнение происходило по содержимому, а не по ссылке на объект. Если оставить только `__eq__`, то объекты будут сравниваться по ссылке, и два разных объекта с одинаковыми координатами будут считаться разными. Это нарушает принцип работы хэш-таблиц, где одинаковые объекты должны иметь одинаковый хэш.

## Задание 2: `@dataclass(frozen=True)` как ключ словаря

Перепиши класс `Point` (двумерная точка) через `@dataclass(frozen=True)`. Поля — `x: int`, `y: int`.

Затем используй точки как **ключи словаря**, чтобы посчитать, сколько раз каждая точка встретилась в списке `points`.

Подсказка: `frozen=True` автоматически генерирует `__hash__` — никаких дополнительных методов писать не нужно. Сравни с заданием 1 по объёму кода.

In [4]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Point():
    x: int
    y: int

points = [
    Point(0, 0), Point(1, 2), Point(0, 0),
    Point(3, 4), Point(1, 2), Point(0, 0),
]
res = {}
for i in points:
    res[i] = res.get(i, 0) + 1
print(res)

{Point(x=0, y=0): 3, Point(x=1, y=2): 2, Point(x=3, y=4): 1}


## Задание 3: Счётчик созданных экземпляров через атрибут класса

Реализуй класс `Widget`, который считает, сколько всего экземпляров было создано за время работы программы. Используй **атрибут класса** `instances_created` как счётчик: при каждом вызове `__init__` увеличивай его на 1.

Класс хранит имя в `self.name`. Метод-классметод `total()` возвращает текущее значение счётчика.

Проверка:
- создай 3 виджета
- `Widget.total()` должно вернуть `3`
- `Widget.instances_created` тоже `3`

In [8]:

# TODO: implement
# your code:
class Widget():
    instances_created = 0

    def __init__(self, name):
        self.name = name
        Widget.instances_created += 1

    @classmethod
    def total(cls):
        return cls.instances_created


w1 = Widget("Widget 1")
w2 = Widget("Widget 2")
w3 = Widget("Widget 3")
print(Widget.total())  # 3
print(Widget.instances_created)  # 3
    

3
3


**Объясни своими словами:** почему мы пишем `Widget.instances_created += 1`, а не `self.instances_created += 1`? Что произойдёт во втором варианте?

Потому что `instances_created` — это атрибут класса, а не экземпляра. Если использовать `self.instances_created += 1`, то Python создаст новый атрибут `instances_created` для конкретного экземпляра, и счётчик будет увеличиваться только для этого экземпляра, а не для всего класса. В итоге, общий счётчик не будет корректно отражать количество созданных объектов.

## Задание 4: Класс `Stack` со sequence-протоколом

Реализуй класс `Stack` (стек, LIFO):

- `__init__()` — пустой стек
- `push(item)` — кладёт элемент наверх
- `pop()` — снимает и возвращает верхний элемент (если пусто — `IndexError`)
- `__len__` — текущая высота стека
- `__getitem__(i)` — индексация, `0` это нижний элемент
- `__repr__` — формат `Stack([1, 2, 3])`

Проверка: благодаря паре `__len__` + `__getitem__` стек должен **автоматически** поддерживать `for`-цикл и `list()` без `__iter__` — это и есть sequence-протокол.

In [12]:

# TODO: implement
# your code:
class Stack:
    def __init__(self):
        self.length = 0
        self.stack = []

    def push(self, item):
        self.stack.append(item)
        self.length += 1

    def pop(self):
        self.stack.pop()
        self.length -= 1

    def __len__(self):
        return self.length

    def __getitem__(self, i):
        return self.stack[i]

    def __repr__(self):
        return f"Stack({self.stack})"


stk = Stack()
stk.push(100)
stk.push(200)
print(len(stk))  # 2
for i in range(len(stk)):
    print(stk[i])  # 100, 200

2
100
200


**Объясни своими словами:** почему `for x in stack:` работает без явного `__iter__`? Что Python делает за нас?

Потому что мы реализовали методы `__len__` и `__getitem__`, которые позволяют Python использовать стек как последовательность (sequence). Когда Python видит, что объект поддерживает эти методы, он автоматически создает итератор для объекта, позволяя использовать его в цикле `for`. Таким образом, мы можем перебирать элементы стека без необходимости явно определять метод `__iter__`.

## Задание 5: Иерархия `Animal` → `Dog` / `Cat` с полиморфизмом

Реализуй три класса:

- `Animal` — `__init__(self, name)`, метод `speak(self)` возвращает строку `"..."` (заглушка)
- `Dog(Animal)` — переопределяет `speak`, возвращает `"гав"`
- `Cat(Animal)` — переопределяет `speak`, возвращает `"мяу"`

Затем напиши **функцию** `chorus(animals)`, которая принимает список животных и возвращает строку вида `"Барсик: мяу; Рекс: гав; Мурка: мяу"`. Функция не должна знать конкретный тип — просто вызывает `animal.speak()`. Это и есть полиморфизм.

In [23]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return '...'


class Dog(Animal):
    def __init__(self, name):
        super().__init__(name)

    def speak(self):
        return 'Гав'


class Cat(Animal):
    def __init__(self, name):
        super().__init__(name)

    def speak(self):
        return 'Мяв'


animals = [Cat("Барсик"), Dog("Рекс"), Cat("Мурка"), Animal('Something')]
for i in animals:
    print(f"{i.name} говорит {i.speak()}")

Барсик говорит Мяв
Рекс говорит Гав
Мурка говорит Мяв
Something говорит ...


## Задание 6: `@classmethod` фабрика — парсинг CSV-строки

Реализуй `@dataclass` `Employee` с полями `name: str`, `salary: int`, `role: str`. Добавь к нему `@classmethod from_csv_row(cls, row)`, который принимает строку формата `"Аня,100000,engineer"` и возвращает новый объект `Employee`.

Подсказка: используй `cls(...)`, а не `Employee(...)` — это правильный паттерн фабрики, который корректно работает при наследовании.

Затем примени фабрику к списку строк через `map`/list-comprehension.

In [24]:
from dataclasses import dataclass


@dataclass
class Employee:
    name: str
    salary: int
    role: str

    @classmethod
    def from_csv_row(cls, row):
        n, s, r = row.split(',')
        return cls(n, s, r)


rows = [
    "Аня,100000,engineer",
    "Боря,120000,manager",
    "Вера,90000,analyst",
]
list(map(lambda x: Employee.from_csv_row(x), rows))

[Employee(name='Аня', salary='100000', role='engineer'),
 Employee(name='Боря', salary='120000', role='manager'),
 Employee(name='Вера', salary='90000', role='analyst')]

**Объясни своими словами:** что произойдёт, если внутри `from_csv_row` написать `Employee(...)` вместо `cls(...)` и потом унаследоваться от `Employee`?

Если внутри `from_csv_row` использовать `Employee(...)`, то при наследовании от `Employee` и вызове метода `from_csv_row` на подклассе, будет создан объект именно класса `Employee`, а не подкласса. Это нарушает принцип фабрики, так как мы хотим, чтобы метод создавал экземпляры того класса, на котором он был вызван. Использование `cls(...)` гарантирует, что будет создан объект текущего класса (подкласса), что позволяет корректно работать с наследованием.

## Задание 7: `@property` с валидацией в сеттере

Реализуй класс `Temperature` с одним полем — температурой в Цельсиях. Используй `@property` для геттера и `@<имя>.setter` для сеттера, чтобы при попытке поставить значение ниже `-273.15` (абсолютный ноль) — поднимать `ValueError`.

Подсказки:
- внутреннее значение храни в `self._celsius` (соглашение: подчёркивание = «приватный»)
- геттер: `@property def celsius(self): return self._celsius`
- сеттер: `@celsius.setter def celsius(self, value): ...` с проверкой
- в `__init__` тоже используй сеттер через `self.celsius = ...`, чтобы валидация сработала и при создании

Проверка: `Temperature(-300)` должно упасть с `ValueError`; `t.celsius = 25` — работать.

In [25]:

# TODO: implement
# your code:
class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError
        self._celsius = value

c = Temperature(25)
print(c.celsius)  # 25
c.celsius = -300  # ValueError

25


ValueError: 

# Готово

Ты только что прошёл задачи на пересечении классов, наследования, магических методов, `@dataclass` и `@classmethod`. Если все ячейки прошли — концепты ООП у тебя работают как единое целое.

На следующей неделе мы разберём память Python, потоки и асинхронность — и увидим, как классы из этой недели становятся базой для async-контекстных менеджеров (`__aenter__`/`__aexit__`) и собственных типов в многопоточных программах.
